# SVM (RBF) Hyperparameter Tuning
Tunes Pabodha's SVM using the existing chronological split. **Validation Macro F1** is the primary selection metric because the target is imbalanced. Test labels are not used for model selection.

In [ ]:
import sys, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

current=Path.cwd().resolve()
project_root=current
for p in [current]+list(current.parents):
    if (p/"data"/"processed").exists() and (p/"src").exists():
        project_root=p; break
if str(project_root) not in sys.path: sys.path.insert(0,str(project_root))
from src.preprocessing.preprocessing_pipeline import prepare_base_dataframe, prepare_features_and_target, build_preprocessor
print("Project root:",project_root)

## 1. Load Chronological Splits

In [ ]:
d=project_root/"data"/"processed"
paths=[d/"train_data.csv",d/"validation_data.csv",d/"test_data.csv"]
for p in paths:
    if not p.exists(): raise FileNotFoundError(p)
types={"site_id":"string","state_code":"string","county_code":"string","site_num":"string"}
train_df=pd.read_csv(paths[0],dtype=types,low_memory=False)
val_df=pd.read_csv(paths[1],dtype=types,low_memory=False)
test_df=pd.read_csv(paths[2],dtype=types,low_memory=False)
print(train_df.shape,val_df.shape,test_df.shape)

## 2. Prepare Features and Fit Preprocessing on Training Only

In [ ]:
train=prepare_base_dataframe(train_df); val=prepare_base_dataframe(val_df); test=prepare_base_dataframe(test_df)
X_train,y_train=prepare_features_and_target(train)
X_val,y_val=prepare_features_and_target(val)
X_test,y_test=prepare_features_and_target(test)
preprocessor=build_preprocessor(X_train)
Xtr=preprocessor.fit_transform(X_train)
Xv=preprocessor.transform(X_val)
Xt=preprocessor.transform(X_test)
print("Processed:",Xtr.shape,Xv.shape,Xt.shape)
print(y_train.value_counts())

## 3. Recalculate Baseline

In [ ]:
baseline=SVC(kernel="rbf",C=1.0,gamma="scale",class_weight=None)
baseline.fit(Xtr,y_train)
bp=baseline.predict(Xv)
baseline_accuracy=accuracy_score(y_val,bp)
baseline_macro_f1=f1_score(y_val,bp,average="macro",zero_division=0)
print(f"Baseline Accuracy: {baseline_accuracy:.6f}")
print(f"Baseline Macro F1: {baseline_macro_f1:.6f}")

## 4. Tune C, gamma, and class_weight
A compact search is used to keep SVM runtime manageable. Every candidate is trained on the historical training split and evaluated on the later validation split.

In [ ]:
grid=[
{"C":0.5,"gamma":"scale","class_weight":None},
{"C":1.0,"gamma":"scale","class_weight":None},
{"C":2.0,"gamma":"scale","class_weight":None},
{"C":5.0,"gamma":"scale","class_weight":None},
{"C":1.0,"gamma":"auto","class_weight":None},
{"C":2.0,"gamma":"auto","class_weight":None},
{"C":0.5,"gamma":"scale","class_weight":"balanced"},
{"C":1.0,"gamma":"scale","class_weight":"balanced"},
{"C":2.0,"gamma":"scale","class_weight":"balanced"},
{"C":5.0,"gamma":"scale","class_weight":"balanced"},
{"C":1.0,"gamma":"auto","class_weight":"balanced"},
{"C":2.0,"gamma":"auto","class_weight":"balanced"},
]
print("Candidates:",len(grid))

In [ ]:
rows=[]
for i,p in enumerate(grid,1):
    print(f"{i}/{len(grid)}",p)
    t=time.time()
    m=SVC(kernel="rbf",**p)
    m.fit(Xtr,y_train)
    pred=m.predict(Xv)
    rows.append({**p,
        "validation_accuracy":accuracy_score(y_val,pred),
        "validation_macro_f1":f1_score(y_val,pred,average="macro",zero_division=0),
        "seconds":time.time()-t})
results=pd.DataFrame(rows).sort_values(["validation_macro_f1","validation_accuracy"],ascending=False).reset_index(drop=True)
results

## 5. Select Best Configuration by Validation Macro F1

In [ ]:
best=results.iloc[0]
best_C=float(best["C"]); best_gamma=best["gamma"]; best_weight=best["class_weight"]
if pd.isna(best_weight): best_weight=None
print("Best C:",best_C)
print("Best gamma:",best_gamma)
print("Best class_weight:",best_weight)
print(f"Best validation Accuracy: {best['validation_accuracy']:.6f}")
print(f"Best validation Macro F1: {best['validation_macro_f1']:.6f}")

## 6. Train and Evaluate Selected Tuned SVM

In [ ]:
tuned=SVC(kernel="rbf",C=best_C,gamma=best_gamma,class_weight=best_weight)
tuned.fit(Xtr,y_train)
tp=tuned.predict(Xv)
tuned_accuracy=accuracy_score(y_val,tp)
tuned_macro_f1=f1_score(y_val,tp,average="macro",zero_division=0)
report=pd.DataFrame(classification_report(y_val,tp,output_dict=True,zero_division=0)).T
print(f"Tuned Accuracy: {tuned_accuracy:.6f}")
print(f"Tuned Macro F1: {tuned_macro_f1:.6f}")
report

## 7. Baseline vs Tuned

In [ ]:
comparison=pd.DataFrame([
{"model":"SVM (RBF) Baseline","validation_accuracy":baseline_accuracy,"validation_macro_f1":baseline_macro_f1},
{"model":"SVM (RBF) Tuned","validation_accuracy":tuned_accuracy,"validation_macro_f1":tuned_macro_f1}])
comparison["accuracy_change_vs_baseline"]=comparison.validation_accuracy-baseline_accuracy
comparison["macro_f1_change_vs_baseline"]=comparison.validation_macro_f1-baseline_macro_f1
comparison

## 8. Save Results and Figures

In [ ]:
results_dir=project_root/"reports"/"model_results"; pics=project_root/"reports"/"pictures"
results_dir.mkdir(parents=True,exist_ok=True); pics.mkdir(parents=True,exist_ok=True)
all_path=results_dir/"svm_rbf_tuning_all_results.csv"
comp_path=results_dir/"svm_rbf_baseline_vs_tuned.csv"
report_path=results_dir/"svm_rbf_tuned_classification_report.csv"
tuned_path=results_dir/"svm_rbf_tuned_results.csv"
params_path=results_dir/"svm_rbf_best_parameters.csv"
results.to_csv(all_path,index=False); comparison.to_csv(comp_path,index=False); report.to_csv(report_path)
pd.DataFrame([{"model":"SVM (RBF) Tuned","validation_accuracy":tuned_accuracy,"validation_macro_f1":tuned_macro_f1}]).to_csv(tuned_path,index=False)
pd.DataFrame([{"C":best_C,"gamma":best_gamma,"class_weight":str(best_weight)}]).to_csv(params_path,index=False)

labels=sorted(pd.Series(y_val).dropna().unique())
fig,ax=plt.subplots(figsize=(9,7))
ConfusionMatrixDisplay.from_predictions(y_val,tp,labels=labels,xticks_rotation=45,ax=ax)
ax.set_title("SVM (RBF) Tuned - Validation Confusion Matrix"); plt.tight_layout()
cm_path=pics/"svm_rbf_tuned_confusion_matrix.png"; plt.savefig(cm_path,dpi=300,bbox_inches="tight"); plt.show()

ax=comparison.set_index("model")[["validation_accuracy","validation_macro_f1"]].plot(kind="bar",figsize=(9,6),rot=0)
ax.set_title("SVM (RBF) - Baseline vs Tuned Validation Performance"); ax.set_ylabel("Score"); ax.set_ylim(0,1); plt.tight_layout()
plot_path=pics/"svm_rbf_baseline_vs_tuned.png"; plt.savefig(plot_path,dpi=300,bbox_inches="tight"); plt.show()

## 9. Output Check

In [ ]:
for p in [all_path,comp_path,report_path,tuned_path,params_path,cm_path,plot_path]:
    print(("OK" if p.exists() else "MISSING"),"-",p)
print("\nIMPORTANT: Selection metric = validation Macro F1.")
print("IMPORTANT: Preprocessor fitted on training data only.")
print("IMPORTANT: Test labels were not used for tuning/model selection.")